In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier,RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier
from tqdm import tqdm

from IPython.display import Image

In [14]:
glass=pd.read_csv("D:\Machine Learning\Datasets\Cases\Glass_Identification\Glass.csv")

In [20]:
from sklearn.preprocessing import OneHotEncoder,LabelEncoder

from sklearn.compose import make_column_selector
from tqdm import tqdm
X, y = glass.drop("Type", axis=1), glass["Type"]
ohe = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")
col_transformer = ColumnTransformer([("OHE", ohe, make_column_selector(dtype_include=object))], 
                                    remainder='passthrough', 
                                    verbose_feature_names_out=False)
col_transformer = col_transformer.set_output(transform="pandas")
X = col_transformer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=25)


In [17]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score
from tqdm import tqdm


trees=[50,100,150,200]
rates=[0.01,0.1,0.2,0.8]
depths=[None,2,3,4,5]
scores=[]
for t in tqdm(trees):
    for d in depths:
        for r in rates:
            gbm=GradientBoostingClassifier(random_state=25,max_depth=d,learning_rate=r,n_estimators=t)
            gbm.fit(X_train,y_train)
            y_pred=gbm.predict(X_test)
            scores.append([t,d,r,f1_score(y_test,y_pred,average='micro')])
df_scores=pd.DataFrame(scores,columns=['trees','depth','learning rate','score'])
df_scores.sort_values('score',ascending=False)

100%|██████████| 4/4 [01:14<00:00, 18.60s/it]


,trees,depth,learning rate,score
15,50,4.0,0.80,0.837209
14,50,4.0,0.20,0.837209
76,200,5.0,0.01,0.837209
74,200,4.0,0.20,0.837209
75,200,4.0,0.80,0.837209
...,...,...,...,...
43,150,NaN,0.80,0.674419
63,200,NaN,0.80,0.674419
8,50,3.0,0.01,0.651163
28,100,3.0,0.01,0.651163


# XG boost

In [18]:
from xgboost import XGBClassifier

In [25]:
glass=pd.read_csv("D:\Machine Learning\Datasets\Cases\Glass_Identification\Glass.csv")

X, y = glass.drop("Type", axis=1), glass["Type"]
le = LabelEncoder()
y = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25, stratify=y)
trees = [50, 100, 150, 200]
rates = [0.01, 0.1, 0.2, 0.5, 0.8]
depths = [None, 2, 3, 4, 5]
scores = []
for t in tqdm(trees):
    for r in rates:
        for d in depths:
            gbm = XGBClassifier(n_estimators=t, learning_rate=r, max_depth=d, random_state=25)
            gbm.fit(X_train, y_train)
            y_pred = gbm.predict(X_test)
            f1 = f1_score(y_test, y_pred, average='macro')
            scores.append([t, d, r, f1])
df_scores = pd.DataFrame(scores, columns=["n_estimators", "max_depth", "learning_rate", "score"])
df_scores.sort_values("score", ascending=False)

100%|██████████| 4/4 [00:15<00:00,  3.81s/it]


,n_estimators,max_depth,learning_rate,score
85,200,NaN,0.20,0.832031
35,100,NaN,0.20,0.824693
60,150,NaN,0.20,0.824693
84,200,5.0,0.10,0.824693
49,100,5.0,0.80,0.817927
...,...,...,...,...
1,50,2.0,0.01,0.580505
28,100,4.0,0.01,0.573782
26,100,2.0,0.01,0.573671
3,50,4.0,0.01,0.565793


# Light GBM

In [28]:
from lightgbm import LGBMClassifier

In [29]:
import warnings
warnings.filterwarnings('ignore')

In [31]:
lgbm=LGBMClassifier(random_state=25,varbose=-1)
lgbm.fit(X_train,y_train)


LGBMClassifier(random_state=25, varbose=-1)

In [32]:
X, y = glass.drop("Type", axis=1), glass["Type"]
le = LabelEncoder()
y = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25, stratify=y)
trees = [50, 100, 150, 200]
rates = [0.01, 0.1, 0.2, 0.5, 0.8]
depths = [None, 2, 3, 4, 5]
scores = []
for t in tqdm(trees):
    for r in rates:
        for d in depths:
            lgbm = LGBMClassifier(n_estimators=t, learning_rate=r, max_depth=d, random_state=25)
            lgbm.fit(X_train, y_train)
            y_pred = lgbm.predict(X_test)
            f1 = f1_score(y_test, y_pred, average='macro')
            scores.append([t, d, r, f1])
df_scores = pd.DataFrame(scores, columns=["n_estimators", "max_depth", "learning_rate", "score"])
df_scores.sort_values("score", ascending=False)

100%|██████████| 4/4 [00:13<00:00,  3.47s/it]


,n_estimators,max_depth,learning_rate,score
40,100,NaN,0.50,0.854124
65,150,NaN,0.50,0.846927
90,200,NaN,0.50,0.846927
66,150,2.0,0.50,0.839529
87,200,3.0,0.20,0.831932
...,...,...,...,...
0,50,NaN,0.01,0.378053
4,50,5.0,0.01,0.378053
3,50,4.0,0.01,0.378053
2,50,3.0,0.01,0.364575


# CatBoost

In [34]:
from catboost import CatBoostClassifier

In [36]:
X, y = glass.drop("Type", axis=1), glass["Type"]
le = LabelEncoder()
y = le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25, stratify=y)
trees = [50, 100, 150, 200]
rates = [0.01, 0.1, 0.2, 0.5, 0.8]
depths = [None, 2, 3, 4, 5]
scores = []
for t in tqdm(trees):
    for r in rates:
        for d in depths:
            cat = CatBoostClassifier(n_estimators=t, learning_rate=r, max_depth=d, random_state=25,verbose=False)
            cat.fit(X_train, y_train)
            y_pred = cat.predict(X_test)
            f1 = f1_score(y_test, y_pred, average='macro')
            scores.append([t, d, r, f1])
df_scores = pd.DataFrame(scores, columns=["n_estimators", "max_depth", "learning_rate", "score"])
df_scores.sort_values("score", ascending=False)

100%|██████████| 4/4 [00:35<00:00,  8.86s/it]


,n_estimators,max_depth,learning_rate,score
20,50,NaN,0.80,0.853940
95,200,NaN,0.80,0.853940
70,150,NaN,0.80,0.853940
45,100,NaN,0.80,0.853940
99,200,5.0,0.80,0.845659
...,...,...,...,...
28,100,4.0,0.01,0.365543
77,200,3.0,0.01,0.365543
2,50,3.0,0.01,0.364659
27,100,3.0,0.01,0.363680


In [39]:
## Gradient Boost Regressor

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error


conc = pd.read_csv("D:\Machine Learning\Datasets\Cases\Concrete_Strength\Concrete_Data.csv")
X, y = conc.drop("Strength", axis=1), conc["Strength"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25)
trees = [50, 100, 150, 200]
rates = [0.01, 0.1, 0.2, 0.5, 0.8]
depths = [None, 2, 3, 4, 5]
scores = []
for t in tqdm(trees):
    for r in rates:
        for d in depths:
            gbm = GradientBoostingRegressor(n_estimators=t, learning_rate=r, max_depth=d, random_state=25, verbose=False)
            gbm.fit(X_train, y_train)
            y_pred = gbm.predict(X_test)
            mae = mean_absolute_error(y_test, y_pred)
            scores.append([t, d, r, mae])
df_scores = pd.DataFrame(scores, columns=["n_estimators", "max_depth", "learning_rate", "score"])
df_scores.sort_values("score", ascending=True)

100%|██████████| 4/4 [00:22<00:00,  5.67s/it]


,n_estimators,max_depth,learning_rate,score
89,200,5.0,0.20,2.952430
64,150,5.0,0.20,2.987907
39,100,5.0,0.20,3.049501
88,200,4.0,0.20,3.059607
63,150,4.0,0.20,3.082901
...,...,...,...,...
26,100,2.0,0.01,9.520785
4,50,5.0,0.01,9.588882
3,50,4.0,0.01,9.849491
2,50,3.0,0.01,10.365419
